# akb × GeoAgent — Interactive Demo

This notebook shows how to connect the Archive Knowledge Base (`akb`) to
GeoAgent so you can query the corpus with natural language and render results
on a live Leaflet map — with optional overlay of satellite / STAC data.

## Install
```bash
pip install -e '../[llm,geoagent]'
python -m spacy download en_core_web_sm
```

## Quickstart pipeline (if not already run)
```bash
akb ingest sample_data/lake_chad_climate.md
akb ingest sample_data/water_harvesting_sahel.md
akb ingest sample_data/kanem_bornu_empire.md
akb ingest sample_data/boko_haram_political_economy.md
akb ingest sample_data/decentralised_finance_africa.md
akb chunk --all && akb embed --all && akb ner --all
akb resolve geo --all && akb resolve chrono --all
```

In [ ]:
import sys, pathlib
# Make sure akb is importable from the notebook
sys.path.insert(0, str(pathlib.Path('..').resolve()))

## 1 · Direct tool calls (no GeoAgent install required)

Every akb tool returns plain dicts / GeoJSON — you can call them without
the full GeoAgent stack and load results directly into leafmap.

In [ ]:
from cli.geoagent_tools import (
    akb_search_locations,
    akb_get_timeline_locations,
    akb_get_entity_network,
    akb_export_geojson,
)

### 1a · Semantic search → map

In [ ]:
import leafmap

geojson = akb_search_locations("water harvesting flood mitigation agricultural yield", top_k=12)
print(f"{len(geojson['features'])} location features")

m = leafmap.Map(center=[12, 10], zoom=5)
m.add_geojson(geojson, layer_name="Water harvesting search")
m

### 1b · Temporal filter — medieval period (900–1500 CE)

In [ ]:
medieval = akb_get_timeline_locations(iso_start="0900", iso_end="1500")
print(f"{len(medieval['features'])} features in 900–1500 CE window")
for f in medieval['features'][:6]:
    p = f['properties']
    print(f"  {p['name']:<35} t={p['time_context']:<20} [{p['block_title']}]")

In [ ]:
m2 = leafmap.Map(center=[12, 13], zoom=5)
m2.add_geojson(medieval, layer_name="Medieval period (900–1500 CE)")
m2

### 1c · Entity network — what does the corpus say about Maiduguri?

In [ ]:
import json
network = akb_get_entity_network("Maiduguri")
print(f"{network['occurrence_count']} chunks mention Maiduguri\n")
for occ in network['occurrences']:
    print(f"[{occ['block_title']}]")
    print(f"  excerpt: {occ['excerpt'][:120]}...")
    for t, vals in occ['co_entities'].items():
        print(f"  {t}: {', '.join(vals[:4])}")
    print()

## 2 · Full GeoAgent session

Requires `pip install geoagent leafmap`. The agent can call any of the four
akb tools autonomously, then chain them with its own spatial tools (layer
management, STAC search, NASA OPERA rasters).

In [ ]:
from cli.geoagent_tools import make_agent

m3 = leafmap.Map(center=[12, 10], zoom=5)
agent = make_agent(map=m3, model="claude-sonnet-4-6")
m3  # display map — agent will add layers to it as it runs

In [ ]:
# Multi-step query: search corpus + overlay satellite water extent
agent.run(
    "Search the knowledge base for locations related to climate displacement and "
    "conflict in northeast Nigeria, add them to the map, then overlay the current "
    "MODIS surface water layer for the Lake Chad basin."
)

In [ ]:
# Temporal deep-dive: Kanem-Bornu period
agent.run(
    "Show all locations from the knowledge base that date to between 900 and 1500 CE, "
    "then tell me which of those locations are within 200 km of Lake Chad."
)

In [ ]:
# Cross-disciplinary synthesis
agent.run(
    "Use the knowledge base to find everything the archive says about water management "
    "in Borno State across all time periods. Summarise the argument chain from "
    "historical Kanem-Bornu canal systems to modern swale rehabilitation proposals, "
    "citing specific sources and dates."
)

## 3 · Export for external GIS tools

All GeoJSON outputs drop straight into QGIS, Google Earth, or Observable.

In [ ]:
import json, pathlib

# Full corpus GeoJSON → QGIS / Google Earth
full = akb_export_geojson()
pathlib.Path("../data/corpus_locations.geojson").write_text(json.dumps(full, indent=2))
print(f"Wrote {len(full['features'])} features to data/corpus_locations.geojson")

# Filtered: Lake Chad documents only
lake_chad = akb_export_geojson(block_title_filter="lake chad")
pathlib.Path("../data/lake_chad_locations.geojson").write_text(json.dumps(lake_chad, indent=2))
print(f"Wrote {len(lake_chad['features'])} features to data/lake_chad_locations.geojson")

## CLI equivalents

Everything above is also runnable from the terminal:

```bash
# Search → GeoJSON stdout
akb geoagent --tool search-locations --query "water harvesting flood"

# Timeline filter → file
akb geoagent --tool timeline-locations --start 0900 --end 1500 --out medieval.geojson

# Entity network
akb geoagent --tool entity-network --query "Maiduguri"

# Full export
akb geoagent --tool export-geojson --out corpus.geojson

# Full agent prompt
akb geoagent "Show water management sites from 900–1500 CE and overlay MODIS water"
```